# Explainability Evaluation - Reconstruction Error Agreement

## Testing the Attention based Explainability and SHAP on Agreement with Reconstruction Error

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Connect and Access Git Repository

import os
import sys
import json
from getpass import getpass
from pathlib import Path

GITHUB_USERNAME = "sharaffatima"
REPO_NAME = "bachelor-thesis"

token = getpass("GitHub token: ")

REPO_URL = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
REPO_DIR = Path("/content") / REPO_NAME


if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull

%cd {REPO_DIR}
sys.path.append(str(REPO_DIR))

print("Repo exists:", REPO_DIR.exists())

GitHub token: ··········
Cloning into '/content/bachelor-thesis'...
remote: Enumerating objects: 687, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (142/142), done.
remote: Total 687 (delta 125), reused 110 (delta 59), pack-reused 485 (from 2)
Receiving objects: 100% (687/687), 36.27 MiB | 21.63 MiB/s, done.
Resolving deltas: 100% (365/365), done.
/content/bachelor-thesis
Repo exists: True


In [4]:
!git pull

Already up to date.


In [6]:
import os
import numpy as np
import pandas as pd
import shap
import torch
from torch.utils.data import DataLoader, TensorDataset

# global configuration

REPO_DIR = Path("/content/bachelor-thesis")
sys.path.append(str(REPO_DIR))
sys.path.append(str(REPO_DIR / "explainability"))


from utils.dataset import PTBXLDataset
from utils.seed import set_seed, RANDOM_SEED
from explainability.anoformer_attention import AnoFormerExplainability
from explainability.tranad_attention import TranADExplainability
from models.anoformer import AnoFormerMainModel
from models.tranad import TranADMainModel
from explainability.evaluation import ReconstructionErrorAgreementEvaluation
from explainability.shap_wrappers import AnoFormerAnomalyScoreWrapper, TranADAnomalyScoreWrapper


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(RANDOM_SEED)
print(f"Active device: {DEVICE}")

Active device: cuda


In [7]:
# Paths to Directories
# Load processed data and pretrained models

DATA_DIR = Path("/content/drive/MyDrive/ECG data")
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = DATA_DIR / "results"

MODEL_DIR = RESULTS_DIR / "models"
ANOFORMER_MODEL_PATH = MODEL_DIR / "anoformer" / "anoformer_40epochs_larger_capacity_2.pt"
TRANAD_MODEL_PATH = MODEL_DIR / "tranad" / "tranad_20epochs_larger_capacity.pt"

SHAP_RESULTS_DIR = RESULTS_DIR / "shap"

AGREEMENT_DIR = RESULTS_DIR / "evaluation" / "reconstruction_error_agreement"
AGREEMENT_DIR .mkdir(parents=True, exist_ok=True)

In [8]:
# Load my preprocessed training and test datasets

X_train = np.load(PROCESSED_DIR / "X_train.npy")
y_train = np.load(PROCESSED_DIR / "y_train.npy")

X_test = np.load(os.path.join(PROCESSED_DIR, "X_test.npy"))
y_test = np.load(os.path.join(PROCESSED_DIR, "y_test.npy"))

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

(17418, 1000, 12) (17418,)
(2198, 1000, 12) (2198,)


In [9]:
# select only normal ecgs from Training set
normal_train_mask = y_train == 0
X_train_normal = X_train[normal_train_mask]
y_train_normal = y_train[normal_train_mask]

# Datasets
# We need the Train dataset because the SHAP background is from the training set. We also need the mean and std of the training set to normalize the test set.
train_dataset = PTBXLDataset(
    X_train_normal,
    y_train_normal,
    normalize=True
)

# We need the Test dataset because the chosen samples for evaluation are from the test set
test_dataset = PTBXLDataset(
    X_test,
    y_test,
    mean=train_dataset.mean,
    std=train_dataset.std,
    normalize=True
)

# Dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    drop_last=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False
)

In [10]:
# Load previosly chosen ECGs for Explainability and Evaluation

SharedSamples = np.load(
    RESULTS_DIR
    / "evaluation"
    / "shared_explanation_samples.npz"
)

NormalIndex = int(SharedSamples["normal_index"])
AnomalousIndex = int(SharedSamples["anomalous_index"])

NormalECG, NormalLabel = test_dataset[NormalIndex]
AnomalousECG, AnomalousLabel = test_dataset[AnomalousIndex]

NormalECG = NormalECG.unsqueeze(0)
AnomalousECG = AnomalousECG.unsqueeze(0)

print("Index of Normal ECG:", NormalIndex, "Label:", int(NormalLabel))
print("Index of Anomalous ECG:", AnomalousIndex, "Label:", int(AnomalousLabel))

print("Normal ECG shape:", NormalECG.shape)
print("Anomalous ECG shape:", AnomalousECG.shape)

Index of Normal ECG: 583 Label: 0
Index of Anomalous ECG: 495 Label: 1
Normal ECG shape: torch.Size([1, 1000, 12])
Anomalous ECG shape: torch.Size([1, 1000, 12])


In [11]:
# Create the Evaluator instance

AgreementEvaluator = ReconstructionErrorAgreementEvaluation()

# AnoFormer

In [12]:
# Create AnoFormer Model

AnoFormer = AnoFormerMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    DiscriminatorLayers=1,
    ForwardDimension=128,
    DropoutRate=0.1,
    GeneratorLearningRate=1e-4,
    DiscriminatorLearningRate=1e-4,
    MaskingRate=0.15,
    AnomalyThreshold=0.75,
    ReconstructionWeight=10.0,
)

# Load the pretrained model
AnoFormer.Load(str(ANOFORMER_MODEL_PATH))

# Create Explainer to extract Attention Values
AnoFormerExplainer = AnoFormerExplainability(AnoFormer)


Model loaded from: /content/drive/MyDrive/ECG data/results/models/anoformer/anoformer_40epochs_larger_capacity_2.pt


In [14]:
# Extract the reconstruction Error for the normal and anomalous ECGs

NormalOriginal, NormalReconstructed, NormalReconstructionError = (
    AnoFormerExplainer.get_reconstruction_error_over_time(
        TimeSeriesWindow=NormalECG,
        DatasetForInverseNorm=test_dataset,
    )
)

AnomalousOriginal, AnomalousReconstructed, AnomalousReconstructionError = (
    AnoFormerExplainer.get_reconstruction_error_over_time(
        TimeSeriesWindow=AnomalousECG,
        DatasetForInverseNorm=test_dataset,
    )
)

print("Normal reconstruction error:", NormalReconstructionError.shape)
print("Anomalous reconstruction error:", AnomalousReconstructionError.shape)


Normal reconstruction error: (1000,)
Anomalous reconstruction error: (1000,)


### Attention Evaluation

In [19]:
# Extract the attention importance values for the normal and anomalous ECGs

NormalAttentionImportance = AnoFormerExplainer.get_attention_importance(
        TimeSeriesWindow=NormalECG,
        BlockIndex=0,
        HeadIndex=None
        )

AnomalousAttentionImportance = AnoFormerExplainer.get_attention_importance(
        TimeSeriesWindow=AnomalousECG,
        BlockIndex=0,
        HeadIndex=None
        )
print("Normal attention importance:", NormalAttentionImportance.shape)
print("Anomalous attention importance:", AnomalousAttentionImportance.shape)

Normal attention importance: (1000,)
Anomalous attention importance: (1000,)


In [15]:
# Evaluate

AnoFormerAttentionNormalAgreement = AgreementEvaluator.evaluate(
    temporal_importance=NormalAttentionImportance,
    reconstruction_error=NormalReconstructionError,
)

AnoFormerAttentionAnomalousAgreement = AgreementEvaluator.evaluate(
    temporal_importance=AnomalousAttentionImportance,
    reconstruction_error=AnomalousReconstructionError,
)

In [16]:
# Print the Results
AnoFormerAgreementResults = pd.DataFrame([
    {
        "Model": "AnoFormer",
        "Explanation": "Attention",
        "Sample": "Normal",
        "Spearman Correlation": AnoFormerAttentionNormalAgreement["spearman_correlation"],
    },
    {
        "Model": "AnoFormer",
        "Explanation": "Attention",
        "Sample": "Anomalous",
        "Spearman Correlation": AnoFormerAttentionAnomalousAgreement["spearman_correlation"],
    }
])

AnoFormerAgreementResults

,Model,Explanation,Sample,Spearman Correlation
0,AnoFormer,Attention,Normal,0.311463
1,AnoFormer,Attention,Anomalous,0.142565


In [18]:
# Save the Results

SavePath = AGREEMENT_DIR / "anoformer_attention_reconstruction_error_agreement_results.csv"

AnoFormerAgreementResults.to_csv(
    SavePath,
    index=False,
)

print("Saved to:", SavePath)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/reconstruction_error_agreement/anoformer_attention_reconstruction_error_agreement_results.csv


## SHAP Evaluation

In [20]:
# Load shap values

anoformer_shap_values = np.load(
    RESULTS_DIR / "shap" / "anoformer" / "anoformer_shap_values.npy"
)

print("Shap values shape:", anoformer_shap_values.shape)

Shap values shape: (2, 1000, 12)


In [21]:
# transform SHAP values into one dimensional intervals

NormalSHAPImportance = np.abs(anoformer_shap_values[0]).sum(axis=1)
AnomalousSHAPImportance = np.abs(anoformer_shap_values[1]).sum(axis=1)

print("Normal shap importance:", NormalSHAPImportance.shape)
print("Anomalous shap importance:", AnomalousSHAPImportance.shape)

Normal shap importance: (1000,)
Anomalous shap importance: (1000,)


In [22]:
# Evaluate

AnoFormerSHAPNormalAgreement = AgreementEvaluator.evaluate(
    temporal_importance=NormalSHAPImportance,
    reconstruction_error=NormalReconstructionError,
)

AnoFormerSHAPAnomalousAgreement = AgreementEvaluator.evaluate(
    temporal_importance=AnomalousSHAPImportance,
    reconstruction_error=AnomalousReconstructionError,
)

In [23]:
# Print the Results
AnoFormerSHAPAgreementResults = pd.DataFrame([
    {
        "Model": "AnoFormer",
        "Explanation": "SHAP",
        "Sample": "Normal",
        "Spearman Correlation": AnoFormerSHAPNormalAgreement["spearman_correlation"],
    },
    {
        "Model": "AnoFormer",
        "Explanation": "SHAP",
        "Sample": "Anomalous",
        "Spearman Correlation": AnoFormerSHAPAnomalousAgreement["spearman_correlation"],
    }
])

AnoFormerSHAPAgreementResults

,Model,Explanation,Sample,Spearman Correlation
0,AnoFormer,SHAP,Normal,0.306519
1,AnoFormer,SHAP,Anomalous,0.335277


In [24]:
# Save the Results

SavePath = AGREEMENT_DIR / "anoformer_shap_reconstruction_error_agreement_results.csv"

AnoFormerSHAPAgreementResults.to_csv(
    SavePath,
    index=False,
)

print("Saved to:", SavePath)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/reconstruction_error_agreement/anoformer_shap_reconstruction_error_agreement_results.csv


# TranAD

In [26]:
# Create TranAD Model

TranAD = TranADMainModel(
    NumFeatures=12,
    ModelDimension=64,
    NumHeads=4,
    NumEncoderLayers=2,
    NumDecoderLayers=2,
    ForwardDimension=128,
    DropoutRate=0.1,
    LearningRate=1e-4,
    FocusGrowthRate=1.05,
    InitialFocusScore=1.0,
    MaxFocusScore=5.0,
    AnomalyPercentile=95.0,
)

# Load the pretrained model
TranAD.Load(str(TRANAD_MODEL_PATH))

# Create Explainer to extract Attention Values
TranADExplainer = TranADExplainability(TranAD)

TranADModel: 268,376 Parameter | d_model=64, H=4, Enc=2, Dec×2=2
Model loaded  /content/drive/MyDrive/ECG data/results/models/tranad/tranad_20epochs_larger_capacity.pt


In [27]:
# Extract the reconstruction Error for the normal and anomalous ECGs

NormalOriginal, NormalReconstructed, NormalReconstructionError = (
    TranADExplainer.get_reconstruction_error_over_time(
        TimeSeriesWindow=NormalECG,
        DatasetForInverseNorm=test_dataset,
    )
)

AnomalousOriginal, AnomalousReconstructed, AnomalousReconstructionError = (
    TranADExplainer.get_reconstruction_error_over_time(
        TimeSeriesWindow=AnomalousECG,
        DatasetForInverseNorm=test_dataset,
    )
)

print("Normal reconstruction error:", NormalReconstructionError.shape)
print("Anomalous reconstruction error:", AnomalousReconstructionError.shape)


Normal reconstruction error: (1000,)
Anomalous reconstruction error: (1000,)


### Attention Evaluation

In [28]:
# Extract Attention Values of Original ECGs

TranADNormalAttentionImportance = TranADExplainer.get_attention_importance(
    TimeSeriesWindow=NormalECG,
    LayerIndex=0,
    HeadIndex=None,
)

TranADAnomalousAttentionImportance = TranADExplainer.get_attention_importance(
    TimeSeriesWindow=AnomalousECG,
    LayerIndex=0,
    HeadIndex=None,
)

print("Normal attention shape:", TranADNormalAttentionImportance.shape)
print("Anomalous attention shape:", TranADAnomalousAttentionImportance.shape)

Normal attention shape: (1000,)
Anomalous attention shape: (1000,)


In [29]:
# Evaluate

TranADAttentionNormalAgreement = AgreementEvaluator.evaluate(
    temporal_importance=NormalAttentionImportance,
    reconstruction_error=NormalReconstructionError,
)

TranADAttentionAnomalousAgreement = AgreementEvaluator.evaluate(
    temporal_importance=AnomalousAttentionImportance,
    reconstruction_error=AnomalousReconstructionError,
)

In [30]:
# Print the Results
TranADAgreementResults = pd.DataFrame([
    {
        "Model": "TranAD",
        "Explanation": "Attention",
        "Sample": "Normal",
        "Spearman Correlation": TranADAttentionNormalAgreement["spearman_correlation"],
    },
    {
        "Model": "TranAD",
        "Explanation": "Attention",
        "Sample": "Anomalous",
        "Spearman Correlation": TranADAttentionAnomalousAgreement["spearman_correlation"],
    }
])

TranADAgreementResults

,Model,Explanation,Sample,Spearman Correlation
0,TranAD,Attention,Normal,0.241966
1,TranAD,Attention,Anomalous,-0.069567


In [31]:
# Save the Results

SavePath = AGREEMENT_DIR / "tranad_attention_reconstruction_error_agreement_results.csv"

TranADAgreementResults.to_csv(
    SavePath,
    index=False,
)

print("Saved to:", SavePath)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/reconstruction_error_agreement/tranad_attention_reconstruction_error_agreement_results.csv


## SHAP Evaluation

In [32]:
# Load shap values

tranad_shap_values = np.load(
    RESULTS_DIR / "shap" / "tranad" / "tranad_shap_values.npy"
)

print("Shap values shape:", tranad_shap_values.shape)

Shap values shape: (2, 1000, 12)


In [33]:
# transform SHAP values into one dimensional intervals

NormalSHAPImportance = np.abs(tranad_shap_values[0]).sum(axis=1)
AnomalousSHAPImportance = np.abs(tranad_shap_values[1]).sum(axis=1)

print("Normal shap importance:", NormalSHAPImportance.shape)
print("Anomalous shap importance:", AnomalousSHAPImportance.shape)

Normal shap importance: (1000,)
Anomalous shap importance: (1000,)


In [34]:
# Evaluate

TranADSHAPNormalAgreement = AgreementEvaluator.evaluate(
    temporal_importance=NormalSHAPImportance,
    reconstruction_error=NormalReconstructionError,
)

TranADSHAPAnomalousAgreement = AgreementEvaluator.evaluate(
    temporal_importance=AnomalousSHAPImportance,
    reconstruction_error=AnomalousReconstructionError,
)

In [35]:
# Print the Results
TranADSHAPAgreementResults = pd.DataFrame([
    {
        "Model": "TranAD",
        "Explanation": "SHAP",
        "Sample": "Normal",
        "Spearman Correlation": TranADSHAPNormalAgreement["spearman_correlation"],
    },
    {
        "Model": "TranAD",
        "Explanation": "SHAP",
        "Sample": "Anomalous",
        "Spearman Correlation": TranADSHAPAnomalousAgreement["spearman_correlation"],
    }
])

TranADSHAPAgreementResults

,Model,Explanation,Sample,Spearman Correlation
0,TranAD,SHAP,Normal,0.569682
1,TranAD,SHAP,Anomalous,0.591290


In [36]:
# Save the Results

SavePath = AGREEMENT_DIR / "tranad_shap_reconstruction_error_agreement_results.csv"

TranADSHAPAgreementResults.to_csv(
    SavePath,
    index=False,
)

print("Saved to:", SavePath)

Saved to: /content/drive/MyDrive/ECG data/results/evaluation/reconstruction_error_agreement/tranad_shap_reconstruction_error_agreement_results.csv
